# Day 100 — Validation Loop & Model Persistence

**Phase 6: Deep Learning & PyTorch**

## 📌 Topics & Learning Objectives
- Decoupling train() and eval() modes
- Disabling gradients with torch.no_grad() during evaluation
- Validation loss and classification accuracy tracking
- Saving and loading state dicts with torch.save and torch.load
- Saving full checkpoint dictionaries (epoch, model state, optimizer state)


### 💡 Concept Overview
In production pipelines, validation evaluating generalization without gradient tracking is essential. `model.eval()` disables Dropout and BatchNorm updates, while `torch.save` checkpoints weights to disk.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Synthetic Data Split
X = torch.randn(500, 4)
y = (X[:, 0] - X[:, 1] > 0).long()
full_ds = TensorDataset(X, y)
train_ds, val_ds = random_split(full_ds, [400, 100])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)


### 1. Dedicated Evaluation Function

In [2]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x_b, y_b in loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            out = model(x_b)
            loss = criterion(out, y_b)
            total_loss += loss.item() * len(x_b)
            correct += (torch.argmax(out, dim=1) == y_b).sum().item()
            total += len(y_b)
    return total_loss / total, (correct / total) * 100.0


### 2. Training with Best Checkpoint Saving

In [3]:
model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 2)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

best_loss = float("inf")
checkpoint_file = "best_model.pth"

for epoch in range(5):
    model.train()
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x_b), y_b)
        loss.backward()
        optimizer.step()

    val_loss, val_acc = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch+1:2d} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.1f}%")
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), checkpoint_file)
        print(f"  -> Saved best weights to {checkpoint_file}")

# Restore
model.load_state_dict(torch.load(checkpoint_file))
print("\nSuccessfully restored best model weights.")


Epoch  1 | Val Loss: 0.3339 | Val Acc: 94.0%
  -> Saved best weights to best_model.pth
Epoch  2 | Val Loss: 0.1882 | Val Acc: 95.0%
  -> Saved best weights to best_model.pth
Epoch  3 | Val Loss: 0.1270 | Val Acc: 100.0%
  -> Saved best weights to best_model.pth
Epoch  4 | Val Loss: 0.1004 | Val Acc: 99.0%
  -> Saved best weights to best_model.pth
Epoch  5 | Val Loss: 0.0855 | Val Acc: 100.0%
  -> Saved best weights to best_model.pth

Successfully restored best model weights.
